# `02-llm.ipynb`
- 기존 기계적인 워크플로우 -> LLM 을 삽입

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# State 정의
from typing import TypedDict, Literal, NotRequired

class TeacherState(TypedDict):
    user_input: str  # 사용자 입력
    sentiment: NotRequired[Literal['positive', 'negative', 'aggressive']]  # 감정 분석
    core_msg:  NotRequired[str]    # 핵심 메세지
    response:  NotRequired[str]    # 최종 답변

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model('gpt-4.1-mini') 

# LLM 초기화 + 감정분석 Node
def analyze_sentiment(state: TeacherState):
    prompt = f'''
    아래 문장의 감정을 분류.
    아래 3가지 중 1가지로 판단
    positive
    negative
    aggressive

    문장: {state['user_input']}
    '''
    result = llm.invoke(prompt)

    return {'sentiment': result.content}  # 바뀐 부분만 return.

In [ ]:
# 나머지 노드
import random

def positivie_node(state: TeacherState):
    msgs = ['최고', '멋져', '훌륭']
    keyword = random.choice(msgs)
    return {'core_msg': keyword}


def negative_node(state: TeacherState):
    msgs = ['힘내', '위로', '괜찮']
    keyword = random.choice(msgs)
    return {'core_msg': keyword}


def aggressive_node(state: TeacherState):
    return {'core_msg':  '공격적인 표현은 삼가라'}

In [9]:
# 완성 Node
def make_final_msg(state: TeacherState):
    prompt = f'''
    다음 사용자 입력에 맞는 답변을 생성해.
    핵심 키워드를 참조해야해.

    사용자 입력: {state['user_input']}
    핵심 키워드: {state['core_msg']}
    '''
    result = llm.invoke(prompt)
    
    return {'response': result.content}

In [8]:
def router(state: TeacherState):
    pass

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TeacherState)
# 노드 등록

# 엣지 등록

app = graph.compile()

In [ ]:
app.invoke({'user_input': '나 너무 우울해'})